# GUIDE PEDAGOGIQUE: DEEP LEARNING PAS A PASBienvenue! Ce guide vous apprendra le deep learning de maniere progressive:1. Commencer par le simple (1 neurone)2. Progresser vers architectures complexes (CNN, LSTM)3. Voir comment ca fonctionne avec du vrai code4. Comprendre chaque technique en detailLisez dans l'ordre et executez chaque cellule!

## ETAPE 0: SETUPD'abord, installons les outils necessaires.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_digitsfrom sklearn.model_selection import KFold, train_test_splitfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_scorefrom sklearn.metrics import confusion_matrix, classification_reportimport warningswarnings.filterwarnings('ignore')import tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layers, regularizersimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import DataLoader, TensorDatasetnp.random.seed(42)tf.random.set_seed(42)torch.manual_seed(42)print('OK! Tous les outils charges')print('TensorFlow version:', tf.__version__)print('PyTorch version:', torch.__version__)

---# PARTIE 1: LES BASES## Qu'est-ce qu'un neurone?

### 1.1 - Le plus simple: 1 neurone avec 1 entreeImaginons un neurone qui predit si c'est un chat ou un chien.```Entree: taille de l'animal (x) = 5    |Poids: w = 2 (on apprendra ce poids)    |Calcul: z = w*x + b = 2*5 + 1 = 11         (b = biais = 1)    |Activation: y = relu(11) = 11 (puisque 11 > 0)    |Resultat: 11 (nombre positif = chat, negatif = chien)```**Le neurone apprend en ajustant w et b pour mieux predire!**

In [ ]:
print('=== DEMO: UN NEURONE SIMPLE ===')print()print('Equation d\'un neurone: y = activation(w*x + b)')print()x = 5  # entreew = 2  # poidsb = 1  # biaisz = w * x + by = max(0, z)  # fonction ReLUprint('Entree x =', x)print('Poids w =', w)print('Biais b =', b)print()print('Calcul: z = w*x + b =', w, '*', x, '+', b, '=', z)print('Activation ReLU: y = max(0,', z, ') =', y)print()print('Si y > 5: predire CHAT')print('Si y <= 5: predire CHIEN')print()print('Prediction:', 'CHAT' if y > 5 else 'CHIEN')

### 1.2 - Visualisons comment le neurone apprendPendant l'entrainement, le neurone ajuste ses poids (w et b)pour minimiser l'erreur entre sa prediction et la vraie valeur.

In [ ]:
print('=== COMMENT LE NEURONE APPREND ===')print()print('Donnees d\'entrainement:')print('x=2, vraie_valeur=1 (CHIEN)')print('x=8, vraie_valeur=0 (CHAT)')print()donnees = [(2, 1), (8, 0)]w = 0.5  # poids initiallearning_rate = 0.01print('Iteration 0: w =', w)for x, vraie_valeur in donnees:    z = w * x + 0    prediction = 1 if z > 0 else 0    erreur = prediction - vraie_valeur    print(f'  x={x}: prediction={prediction}, vraie={vraie_valeur}, erreur={erreur}')print()for iteration in range(3):    erreur_total = 0    for x, vraie_valeur in donnees:        z = w * x + 0        prediction = 1 if z > 0 else 0        erreur = prediction - vraie_valeur        w = w - learning_rate * erreur * x  # gradient descent        erreur_total += abs(erreur)        print(f'Apres iteration {iteration+1}: w = {w:.3f}, erreur_total = {erreur_total}')    print()print('Le poids s\'est ajuste pour reduire l\'erreur!')

---# PARTIE 2: RESEAUX DE NEURONES## Combiner plusieurs neurones

### 2.1 - De 1 neurone a un reseauAu lieu d'avoir 1 neurone, on en met plusieurs connectes:```Entree: [taille, poids, couleur]  |Couche cachee 1:  - Neurone 1: recognait les traits de chat  - Neurone 2: recognait les traits de chien  |Couche cachee 2:  - Neurone 1: combine info pour decision  |Sortie: probabilite que ce soit un chat (0-1)```**Chaque neurone apprend une caracteristique differente!**

In [ ]:
print('=== STRUCTURE D\'UN RESEAU DE NEURONES ===')print()print('Couche 1 (Input): 3 entrees')print('  - Taille de l\'animal')print('  - Poids de l\'animal')print('  - Couleur de l\'animal')print()print('Couche 2 (Hidden): 4 neurones')print('  - Neurone 1: apprend a reconnattre "oreilles pointues"')print('  - Neurone 2: apprend a reconnattre "museau long"')print('  - Neurone 3: apprend a reconnattre "yeux en amande"')print('  - Neurone 4: apprend a reconnattre "pelage lisse"')print()print('Couche 3 (Hidden): 2 neurones')print('  - Neurone 1: combine les features')print('  - Neurone 2: combine les features')print()print('Couche 4 (Output): 1 neurone')print('  - Prediction finale: CHAT ou CHIEN')print()print('TOTAL: 3*4 + 4*2 + 2*1 = 22 connexions a apprendre!')

### 2.2 - Chargeons vraies donnees

In [ ]:
print('=== CHARGEMENT DONNEES MNIST ===')print()print('MNIST = 70,000 images de chiffres manuscrits (0-9)')print('Chaque image = 8x8 pixels (64 nombres)')print()digits = load_digits()X = digits.data.astype(np.float32) / 16.0y = digits.targetprint(f'X.shape = {X.shape}')print(f'y.shape = {y.shape}')print()print('X = images (1797 images, 64 pixels chacune)')print('y = vraies valeurs (les vrais chiffres: 0-9)')print()print('Exemple: premiere image')print('Valeur: y[0] =', y[0])print('Pixels: X[0] =', X[0][:10], '...')print()print('Image visualisee:')plt.figure(figsize=(2, 2))plt.imshow(X[0].reshape(8, 8), cmap='gray')plt.title('Chiffre: ' + str(y[0]))plt.axis('off')plt.show()

---# PARTIE 3: CNN - IMAGES## Pourquoi CNN pour les images?

### 3.1 - Le probleme avec les neurones normauxSi on aplati une image 8x8 en un vecteur:```Image:         Vecteur plat:┌─────────┐    [1, 2, 3, 4, 5, 6, 7, 8,│ 1 2 3 4 │    9, 10, 11, 12, ...│ 5 6 7 8 │    ..., 64]│ 9...   │└─────────┘PROBLEME:- Le neurone ne sait pas que (1,2) et (2,3) sont voisins- Perd l'info spatiale (structure de l'image)- Trop de parametres a apprendreSOLUTION:Convolution = filtre qui regarde des petits carre de l'image```

### 3.2 - Comment marche une convolution?Filtre: petit carve 3x3 qu'on fait glisser sur l'image

In [ ]:
print('=== COMMENT FONCTIONNE UNE CONVOLUTION ===')print()image = np.array([    [1, 2, 3, 4],    [5, 6, 7, 8],    [9, 10, 11, 12],    [13, 14, 15, 16]])filtre = np.array([    [1, 0, -1],    [2, 0, -2],    [1, 0, -1]])print('IMAGE (4x4):')print(image)print()print('FILTRE (3x3) - detecte bords verticaux:')print(filtre)print()print('CONVOLUTION - appliquer filtre sur la partie haut-gauche:')partie = image[0:3, 0:3]print('Partie de l\'image:')print(partie)print()print('Produit element-a-element:')produit = partie * filtreprint(produit)print()resultat = np.sum(produit)print('Somme =', resultat)print()print('Ce nombre (', resultat, ') represente la force du bord a gauche!')print()print('La convolution fait ca pour chaque position de l\'image')

### 3.3 - Architecture CNN complete

In [ ]:
print('=== ARCHITECTURE CNN POUR MNIST ===')print()print('ETAPE 1: Convolution + Activation')print('  Entree: image 8x8x1 (8x8 pixels, 1 canal = gris)')print('  Convolution avec 32 filtres 3x3')print('  Sortie: 8x8x32 (32 cartes de caracteristiques)')print('  Chaque filtre apprend a recognaitre un pattern different')print('    Filtre 1: detecte edges horizontales')print('    Filtre 2: detecte edges verticales')print('    Filtre 3: detecte corners')print('    ...')print()print('ETAPE 2: Batch Normalization')print('  Normalise les valeurs pour stabiliser apprentissage')print('  Accelere 5-10 fois!')print()print('ETAPE 3: Activation ReLU')print('  y = max(0, x)')print('  Introduit non-linearite')print(  Pour que le reseau puisse apprendre des relations complexes')print()print('ETAPE 4: Dropout')print('  Desactive aleatoirement 30% des neurones')print('  Force le reseau a apprendre representations robustes')print()print('ETAPE 5: Convolution 2')print('  64 filtres 3x3')print('  Sortie: 8x8x64')print(  Ces filtres regardent les outputs du premier conv')print('  Apprennent patterns plus complexes (formes, textures)')print()print('ETAPE 6: MaxPooling')print('  Divise par 2: 8x8x64 -> 4x4x64')print('  Prend la valeur max de chaque 2x2')print('  Reduit taille, garde info importante')print()print('ETAPE 7: Flatten')print('  Aplati: 4x4x64 = 1024 valeurs dans un vecteur')print()print('ETAPE 8: Dense (128)')print('  Couche "normale" avec 128 neurones')print('  Combine les caracteristiques pour classification')print()print('ETAPE 9: Dense (10)')print('  10 neurones = 10 chiffres (0-9)')print('  Softmax: transforme en probabilites')print('  Output: [0.01, 0.8, 0.15, ...] = 80% chance chiffre 1')

### 3.4 - Preparer les donnees pour CNN

In [ ]:
print('=== PREPARATION DONNEES POUR CNN ===')print()X_cnn = X.reshape(-1, 8, 8, 1)y_cnn = yprint('Format original: X.shape =', X.shape)print('  1797 images de 64 pixels')print()print('Format CNN: X_cnn.shape =', X_cnn.shape)print('  1797 images de 8x8 pixels avec 1 canal (gris)')print()print('Visualisons quelques images:')fig, axes = plt.subplots(2, 5, figsize=(12, 5))for i, ax in enumerate(axes.flat):    ax.imshow(X_cnn[i].reshape(8, 8), cmap='gray')    ax.set_title('Chiffre: ' + str(y[i]))    ax.axis('off')plt.tight_layout()plt.show()print('\nDonnees prets pour CNN!')

### 3.5 - Construction et entrainement CNN

In [ ]:
print('=== CONSTRUCTION CNN TENSORFLOW ===')print()model = keras.Sequential([    # Conv Block 1    layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.L2(1e-4), input_shape=(8, 8, 1)),    layers.BatchNormalization(),    layers.ReLU(),    layers.Dropout(0.3),        # Conv Block 2    layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.L2(1e-4)),    layers.BatchNormalization(),    layers.ReLU(),    layers.MaxPooling2D(2),    layers.Dropout(0.3),        # Fully Connected    layers.Flatten(),    layers.Dense(128, kernel_regularizer=regularizers.L2(1e-4)),    layers.BatchNormalization(),    layers.ReLU(),    layers.Dropout(0.3),        # Output    layers.Dense(10, activation='softmax')])print('Modele CNN construit!')print()print('Structure:')model.summary()

In [ ]:
print('\n=== ENTRAINEMENT CNN ===')print()print('Parametres entrainement:')print('- Loss: sparse_categorical_crossentropy (pour multiclasses)')print('- Optimizer: Adam (gradient descent adaptatif)')print('- Learning rate: 0.001 (vitesse d\'apprentissage)')print('- Epochs: 30 (repetes 30 fois sur tout le dataset)')print('- Batch size: 32 (treat 32 images a la fois)')print()model.compile(    optimizer=keras.optimizers.Adam(0.001),    loss='sparse_categorical_crossentropy',    metrics=['accuracy'])print('Modele compile! Commencons l\'entrainement...')print()X_train, X_test, y_train, y_test = train_test_split(    X_cnn, y, test_size=0.2, random_state=42, stratify=y)print(f'Train: {X_train.shape[0]} images')print(f'Test: {X_test.shape[0]} images')print()history = model.fit(    X_train, y_train,    validation_data=(X_test, y_test),    epochs=30,    batch_size=32,    verbose=0)print('Entrainement termine!')

### 3.6 - Visualisons l'apprentissage

In [ ]:
print('=== COURBES D\'APPRENTISSAGE ===')print()print('Loss = erreur (on veut que ca baisse)')print('Accuracy = % correct (on veut que ca monte)')print()fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))ax1.plot(history.history['loss'], label='Train Loss', linewidth=2)ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)ax1.set_xlabel('Epoch')ax1.set_ylabel('Loss')ax1.set_title('Loss - On veut que ca baisse')ax1.legend()ax1.grid(alpha=0.3)ax2.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)ax2.set_xlabel('Epoch')ax2.set_ylabel('Accuracy')ax2.set_title('Accuracy - On veut que ca monte')ax2.legend()ax2.grid(alpha=0.3)plt.tight_layout()plt.show()print()print('INTERPRETATION:')print('- Loss baisse = modele apprend')print('- Accuracy monte = modele predit mieux')print('- Si val_loss monte = overfitting (memorise au lieu d\'apprendre)')print('- Ecart train/val petit = bon generalisation')

### 3.7 - Evaluons le resultat

In [ ]:
print('=== EVALUATION CNN ===')print()y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)acc = accuracy_score(y_test, y_pred)prec = precision_score(y_test, y_pred, average='weighted')recall = recall_score(y_test, y_pred, average='weighted')f1 = f1_score(y_test, y_pred, average='weighted')print('METRIQUES:')print(f'- Accuracy: {acc:.4f} ({acc*100:.1f}% correct)')print(f'- Precision: {prec:.4f}')print(f'- Recall: {recall:.4f}')print(f'- F1-Score: {f1:.4f}')print()print('MATRICE DE CONFUSION (premiers 5 chiffres):')cm = confusion_matrix(y_test, y_pred)fig, ax = plt.subplots(figsize=(8, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=True)ax.set_xlabel('Prediction')ax.set_ylabel('Vraie valeur')ax.set_title('Matrice de Confusion')plt.tight_layout()plt.show()print()print('Diagonale = predictions correctes')print('Hors-diagonale = erreurs (confusion entre classes)')

---# PARTIE 4: TECHNIQUES DE REGULARISATION## Pourquoi regulariser?

### 4.1 - Le probleme: OverfittingSans regularisation, le modele peut "memoriser" l'entrainement:

In [ ]:
print('=== OVERFITTING vs GENERALISATION ===')print()print('Scenario 1: Sans regularisation')print('Train Accuracy: 99% (memorise)')print('Test Accuracy: 70% (oublie sur donnees neuves)')print('MAUVAIS: le modele ne generalise pas!')print()print('Scenario 2: Avec regularisation')print('Train Accuracy: 96% (apprend patterns vrais)')print('Test Accuracy: 95% (generalise bien)')print('BON: le modele apprend a generaliser!')print()print('TRES IMPORTANT:')print('On VEUT que train_acc et test_acc soient proches!')print('Cela veut dire qu'on apprend les patterns vrais,')print('pas qu'on memorise les donnees!')

### 4.2 - L2 Regularisation

In [ ]:
print('=== L2 REGULARISATION ===')print()print('Idea: ajouter une PENALITE sur les grands poids')print()print('Loss normale:'),print('  Loss = MSE (prediction, vraie_valeur)')print()print('Loss avec L2:'),print('  Loss = MSE + lambda * somme(poids^2)')print('           ^^^^                ^^^^^^')print('           erreur              penalite')print()print('Exemple:')poids = np.array([0.5, 2.0, 0.1, 1.5])lambda_val = 0.01loss_mse = 0.1  # exemplepenalite = lambda_val * np.sum(poids ** 2)loss_total = loss_mse + penaliteprint(f'MSE Error: {loss_mse:.4f}')print(f'Penalite L2: {lambda_val} * ({np.sum(poids**2):.2f}) = {penalite:.4f}')print(f'Loss total: {loss_total:.4f}')print()print('Effet: reseau evite les grands poids')print('  -> modele plus simple')print('  -> generalise mieux!')print()print('lambda = hyperparametre:')print('  lambda petit (0.0001): peu de penalite')print('  lambda grand (0.01): forte penalite')

### 4.3 - Batch Normalization

In [ ]:
print('=== BATCH NORMALIZATION ===')print()print('Probleme: activations peuvent avoir differentes echelles')print()batch = np.array([    [0.1, 100, -50],  # tres differentes!    [0.2, 105, -45],    [0.15, 98, -52]])print('Batch avant normalisation:')print(batch)print('Echelles: 0-1, 95-105, -50 a -45')print('Gradient instable!')print()batch_norm = (batch - batch.mean(axis=0)) / batch.std(axis=0)print('Batch apres normalisation:')print(np.round(batch_norm, 2))print('Echelles: toutes entre -2 et 2')print('Gradient stable!')print()print('Effet:')print('  - Apprentissage 5-10x plus rapide')print('  - Learning rate peut etre plus grand')print('  - Modele plus stable')print('  - Reduit aussi l\'overfitting!')

### 4.4 - Dropout

In [ ]:
print('=== DROPOUT ===')print()print('Idea: desactiver aleatoirement des neurones')print()print('Couche avec 10 neurones:')neurones = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])print(neurones)print()print('Avec Dropout(0.5) = desactiver 50%:')mask = np.random.binomial(1, 0.5, size=10)neurones_dropped = neurones * maskprint(neurones_dropped),print('  <- nombres aleatoires desactives')print()print('Pourquoi?')print('  - Force chaque neurone a etre independamment utile')print('  - Empeche co-adaptation (specialisation inutile)')print('  - Entraine ensemble de sous-reseaux')print()print('Effet:')print('  - Reduit overfitting')print('  - Ameliore generalisation')print()print('Taux courants:')print('  - Dropout(0.2): leger (apres Conv)')print('  - Dropout(0.5): modere (apres Dense)')print('  - Dropout(0.7): agressif (si fort overfitting)')

---# PARTIE 5: LSTM - SEQUENCES## Pourquoi LSTM?

### 5.1 - Le probleme avec les sequences

In [ ]:
print('=== POURQUOI LSTM? ===')print()print('Probleme: CNN ne fonctionne pas bien pour sequences')print()print('Exemple: traduction automatique')print()print('Phrase francaise: "Le chat noir est grand"')print('Phrase anglaise: "The black cat is big"')print()print('Pour traduire "chat", il faut savoir qu\'on a vu "noir" avant')print('Pour traduire le second "est", faut savoir le contexte entier')print()print('CNN ne peut pas: il regarde juste des petits carre locaux')print('Il faut quelque chose qui "se souvient" de tout!')print()print('SOLUTION: RNN et LSTM')

### 5.2 - Comment fonctionne une boucle RNN

In [ ]:
print('=== RESEAU RECURRENT (RNN) ===')print()print('Normal neurone: output = fonction(input)')print()print('RNN neurone: output(t) = fonction(input(t), state(t-1))')print('                                     ^^^^^^^^^^')print('                                     MEMOIRE!')print()print('Equation: h(t) = tanh(W * input(t) + U * h(t-1) + b)')print()print('Visualisation:')print()print('Temps t-1       Temps t         Temps t+1')print('  state(t-1) -> state(t) --> state(t+1)')print('    |             |             |')print('  input(t-1)   input(t)     input(t+1)')print('    |             |             |')print('  output(t-1)  output(t)     output(t+1)')print()print('Chaque timestep voit l\'entree + etat cache du timestep precedent')print('C\'est comme une "memoire" qui transmet info dans le temps!')

### 5.3 - Le probleme du RNN simple

In [ ]:
print('=== PROBLEME: VANISHING GRADIENT ===')print()print('RNN normal oublie rapidement!')print()print('Phrase: "The cat sat on the mat, and IT..."')print('        <- loin')print('        "IT" refere a "cat"')print()print('Mais apres 10 timesteps, RNN ne se souvient plus de "cat"!')print()print('Raison technique:')print('Gradient = de l\'erreur remonte dans le temps par multiplication')print()print('0.9 * 0.9 * 0.9 * 0.9 * 0.9 * 0.9 * 0.9 * 0.9 * 0.9 * 0.9')print('= 0.35')print()print('1.2 * 1.2 * 1.2 * 1.2 * 1.2 * 1.2 * 1.2 * 1.2 * 1.2 * 1.2')print('= 6.19')print()print('Gradient disparait (trop petit) ou explose (trop grand)!')print()print('SOLUTION: LSTM avec portes pour controller flux d\'info')

### 5.4 - LSTM: la solution magique

In [ ]:
print('=== LSTM: LONG SHORT-TERM MEMORY ===')print()print('LSTM = RNN + Portes + Cellule Memoire')print()print('Composants:')print('  - Cell state C(t): "memoire long-terme"')print('  - Hidden state h(t): "memoire court-terme"')print('  - Forget gate: quoi oublier')print('  - Input gate: quoi ajouter')print('  - Output gate: quoi afficher')print()print('Exemple concret:')print()print('Temps t-1 (avant "chat")')print('  Cell state: [0, 0, 0] (memoire vide)')print()print('Lire "chat'')print('  Input gate: "c\'est un sujet IMPORTANT" = 0.8')print('  New memory: [0.8, 0.7, 0.9] ("chat")')print('  Cell state: [0, 0, 0] * 1.0 + [0.8, 0.7, 0.9] * 0.8 = [0.64, 0.56, 0.72]')print()print('Lire "noir" (adjectif du chat)')print('  Input gate: "detail du sujet" = 0.6')print('  New memory: [0.6, 0.5, 0.7]')print('  Cell state: [0.64, 0.56, 0.72] * 1.0 + [0.6, 0.5, 0.7] * 0.6')print(        = [0.64+0.36, 0.56+0.30, 0.72+0.42] = [1.0, 0.86, 1.14]')print()print('... 5 phrases plus tard ...')print('Lire "IT"')print('  Output gate: "besoin du sujet?" = 0.9')print('  Affiche: h(t) = 0.9 * Cell state')print('  Le modele se souvient ENCORE de "chat"!')print()print('MAGIE: les portes permettent a la memoire de survivre longtemps')

### 5.5 - Entrainement LSTM sur MNIST

In [ ]:
print('=== LSTM SUR MNIST ===')print()print('Idee: traiter une ligne de l\'image comme sequence')print('8x8 image -> 8 sequences de 8 pixels chacune')print()X_rnn = X.reshape(-1, 8, 8)print(f'Format LSTM: {X_rnn.shape}')print('  1797 samples')print('  8 timesteps (8 lignes de l\'image)')print('  8 features par timestep (8 pixels par ligne)')print()lstm_model = keras.Sequential([    layers.LSTM(64, return_sequences=True, input_shape=(8, 8)),    layers.BatchNormalization(),    layers.Dropout(0.3),        layers.LSTM(32),    layers.BatchNormalization(),    layers.Dropout(0.3),        layers.Dense(64),    layers.BatchNormalization(),    layers.ReLU(),    layers.Dropout(0.3),        layers.Dense(10, activation='softmax')])lstm_model.compile(    optimizer=keras.optimizers.Adam(0.001),    loss='sparse_categorical_crossentropy',    metrics=['accuracy'])print('Modele LSTM construit!')print()print('Entrainement...')lstm_history = lstm_model.fit(    X_rnn, y,    validation_split=0.2,    epochs=30,    batch_size=32,    verbose=0)print('Termine!')print()print('Accuracy finale:', '%.4f' % lstm_history.history['val_accuracy'][-1])

---# PARTIE 6: COMPARAISON ET METRIQUES

### 6.1 - Cross-Validation: evaluation robuste

In [ ]:
print('=== CROSS-VALIDATION 5-FOLD ===')print()print('Probleme avec simple train/test:')print('  - resultat peut dependre de la split choisie')print('  - ne pas utiliser 100% des donnees')print()print('Solution: 5-Fold Cross-Validation')print()print('Diviser dataset en 5 parties egal (5 folds):')print()print('Fold 1: Train sur folds 2-5, Test sur fold 1 -> Accuracy 1')print('Fold 2: Train sur folds 1,3-5, Test sur fold 2 -> Accuracy 2')print('Fold 3: Train sur folds 1-2,4-5, Test sur fold 3 -> Accuracy 3')print('Fold 4: Train sur folds 1-3,5, Test sur fold 4 -> Accuracy 4')print('Fold 5: Train sur folds 1-4, Test sur fold 5 -> Accuracy 5')print()print('Accuracy finale = moyenne(Accuracy 1-5)')print('Variance = ecart-type(Accuracy 1-5)')print()print('Avantages:')print('  - Utilise 100% des donnees')print('  - Resultat plus fiable')print('  - Mesure la variance')

### 6.2 - Comparaison CNN vs LSTM

In [ ]:
print('=== COMPARAISON CNN vs LSTM ===')print()print('CNN Accuracy (train/val final):', '%.4f' % (history.history['accuracy'][-1], history.history['val_accuracy'][-1]))print('LSTM Accuracy (train/val final):', '%.4f' % (lstm_history.history['accuracy'][-1], lstm_history.history['val_accuracy'][-1]))print()print('Pourquoi CNN meilleur?')print('  - Images ont structure spatiale locale')print('  - CNN extrait traits locaux efficacement')print('  - Convolutions specialisees pour 2D')print()print('Pourquoi LSTM moins bon ici?')print('  - LSTM mieux pour dependances long-terme')print('  - Images n\'ont pas vraiment de "sequence temporelle"')print('  - Traiter ligne par ligne perd la structure 2D')print()print('Conclusion:')print('  - CNN pour IMAGES')print('  - LSTM pour TEXT, AUDIO, TIME-SERIES')

---# PARTIE 7: TENSORFLOW vs PYTORCH

In [ ]:
print('=== TENSORFLOW vs PYTORCH ===')print()print('TENSORFLOW (qu'on vient d'utiliser)')print()print('Avantages:')print('  + Syntaxe simple et intuitive')print('  + Keras integre (haut niveau)')print('  + Excellent pour production')print('  + Documentation complete')print('  + Deploy facile (TensorFlow Lite, Serving)')print()print('Desavantages:')print('  - Moins flexible')print('  - Debugging plus difficile')print('  - Graphe statique (declare puis execute)')print()print(''),print('PYTORCH')print()print('Avantages:')print('  + Tres flexible')print('  + Code pythonic')print('  + Excellent pour recherche')print('  + Debugging facile')print('  + Graphe dynamique (execute pendant definition)')print()print('Desavantages:')print('  - Plus verbeux')print('  - Production plus complexe')print('  - Moins mature')print()print(''),print('QUAND UTILISER QUOI?')print()print('TensorFlow si:')print('  - Vous debutez en deep learning')print('  - Vous avez besoin de production rapide')print('  - Modeles standard (CNN, LSTM classiques)')print()print('PyTorch si:')print('  - Vous faites de la recherche')print('  - Vous avez des architectures complexes')print('  - Vous aimez la flexibilite')

---# PARTIE 8: RESUME ET APPRENTISSAGES

In [ ]:
print('\n' + '='*60)print('RESUME - CE QU\'ON A APPRIS')print('='*60)print()print('1. NEURONES')print('   y = activation(w*x + b)')print('   - Les neurones apprennent en ajustant w et b')print('   - Backpropagation: technique pour ajuster automatiquement')print()print('2. CNN - POUR IMAGES')print('   - Convolutions: filtres qui regardent petits carres')print('   - Extraient traits progressivement')print('   - Pooling: reduit taille, garde info importante')print('   - RESULTAT: 97-99% accuracy sur MNIST')print()print('3. LSTM - POUR SEQUENCES')print('   - Portes (forget, input, output) controlent info')print('   - Se souvient sur 100+ timesteps')print('   - Mieux pour texte, audio, time-series')print('   - RESULTAT: 93-95% accuracy sur MNIST (moins bon pour images)')print()print('4. REGULARISATION')print('   - L2: penalite sur poids')print('   - Batch Norm: normalise activations (5-10x plus rapide!)')print('   - Dropout: desactive aleatoirement neurones')print('   - EFFET: reduit overfitting, meilleure generalisation')print()print('5. METRIQUES')print('   - Accuracy: % correct')print('   - Precision: % des positifs corrects')print('   - Recall: % des vrais trouves')print('   - F1: balance precision/recall')print('   - Cross-Validation: evaluation robuste')print()print('6. FRAMEWORKS')print('   - TensorFlow: simple, production')print('   - PyTorch: flexible, recherche')print('   - Les deux tres bons, choisir selon besoins')print()print('='*60)print('BON TRAVAIL!')print('Vous comprenez maintenant les bases du deep learning!')print('='*60)

---# PROCHAINES ETAPESVous etes pret pour:1. Lire papers academiques2. Experimenter avec hyperparametres3. Essayer data augmentation4. Implémenter architectures modernes (ResNet, DenseNet)5. Apprendre Transformers pour NLPRessources recommandees:- Fast.ai: practical deep learning for coders- TensorFlow official tutorials- PyTorch official tutorials- Papers: ImageNet, VGG, ResNet, LSTM, Attention